# Propiedades geométricas de contornos

En este cuaderno vas a medir propiedades geométricas sobre formas simples. El objetivo es que cada medida tenga una interpretación clara antes de llevarla a escenas más complejas.


## Objetivo

Calcular área, perímetro, caja contenedora, centroide y cantidad aproximada de vértices sobre contornos bien controlados.

## Resultados de aprendizaje

Al finalizar este cuaderno, vas a poder:

- calcular medidas geométricas básicas de un contorno;
- interpretar qué describe cada medida;
- relacionar la forma observada con la cantidad de vértices aproximados;
- preparar el terreno para analizar objetos reales.

## Relación con la secuencia

Este cuaderno continúa el trabajo sobre contornos. Primero se aprende a detectar el borde; ahora se pasa a describirlo con medidas.


## Módulos que vamos a usar

- `cv2`: para detectar contornos y calcular medidas geométricas.
- `numpy`: para construir un lienzo binario.
- `matplotlib.pyplot`: para mostrar anotaciones.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


## 1. Construir un caso controlado

Vamos a trabajar con un rectángulo, un círculo y un triángulo sobre fondo negro. Eso permite que cada medida se lea con mucha más claridad que en una escena real con ruido.


In [ ]:
lienzo = np.zeros((320, 520), dtype=np.uint8)
cv2.rectangle(lienzo, (30, 60), (170, 250), 255, -1)
cv2.circle(lienzo, (290, 160), 70, 255, -1)
vertices_triangulo = np.array([[400, 250], [500, 250], [450, 80]], dtype=np.int32)
cv2.fillPoly(lienzo, [vertices_triangulo], 255)

plt.figure(figsize=(9, 6), constrained_layout=True)
plt.imshow(lienzo, cmap="gray")
plt.title("Lienzo binario con tres formas", fontweight="bold", loc="left")
plt.axis("off")
plt.show()


## 2. Medir cada contorno

Ahora vamos a detectar los contornos y a calcular varias propiedades. La idea es que cada medida tenga una lectura geométrica concreta.


In [ ]:
contornos, _ = cv2.findContours(lienzo.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contornos = sorted(contornos, key=cv2.contourArea, reverse=True)

def medir_contorno(contorno):
    area = cv2.contourArea(contorno)
    perimetro = cv2.arcLength(contorno, True)
    x, y, ancho, alto = cv2.boundingRect(contorno)

    momentos = cv2.moments(contorno)
    centro_x = int(momentos["m10"] / momentos["m00"])
    centro_y = int(momentos["m01"] / momentos["m00"])

    epsilon = 0.03 * perimetro
    aproximacion = cv2.approxPolyDP(contorno, epsilon, True)

    return {
        "area": area,
        "perimetro": perimetro,
        "caja": (x, y, ancho, alto),
        "centroide": (centro_x, centro_y),
        "vertices_aproximados": len(aproximacion),
    }

mediciones = [medir_contorno(contorno) for contorno in contornos]


In [ ]:
imagen_anotada = cv2.cvtColor(lienzo, cv2.COLOR_GRAY2RGB)

for indice, (contorno, medicion) in enumerate(zip(contornos, mediciones), start=1):
    x, y, ancho, alto = medicion["caja"]
    centro_x, centro_y = medicion["centroide"]
    cv2.drawContours(imagen_anotada, [contorno], -1, (255, 140, 0), 2)
    cv2.rectangle(imagen_anotada, (x, y), (x + ancho, y + alto), (0, 255, 0), 2)
    cv2.circle(imagen_anotada, (centro_x, centro_y), 4, (255, 255, 0), -1)
    cv2.putText(imagen_anotada, f"F{indice}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

plt.figure(figsize=(9, 6), constrained_layout=True)
plt.imshow(imagen_anotada)
plt.title("Contornos con caja y centroide", fontweight="bold", loc="left")
plt.axis("off")
plt.show()


In [ ]:
for indice, medicion in enumerate(mediciones, start=1):
    print(f"Forma {indice}")
    print(f"  Área: {medicion['area']:.1f} píxeles cuadrados")
    print(f"  Perímetro: {medicion['perimetro']:.1f} píxeles")
    print(f"  Caja: {medicion['caja']}")
    print(f"  Centroide: {medicion['centroide']}")
    print(f"  Vértices aproximados: {medicion['vertices_aproximados']}")
    print()


La cantidad de vértices aproximados no identifica mágicamente la figura, pero sí da una pista geométrica útil. En una escena simple, esa información se interpreta rápido. En una escena real, suele combinarse con otras medidas.


## Actividad breve

Agregá una cuarta forma al lienzo y repetí la medición. Después describí:

1. cómo cambió el área;
2. qué te dice la caja contenedora;
3. si la aproximación por vértices ayuda o no a reconocer la forma.


## Cierre

Medir un contorno es una forma de pasar del borde visible a una descripción cuantitativa. Esa transición es importante porque permite comparar objetos, clasificarlos y construir reglas más precisas sobre la forma observada.
